# Распознавание кода Морзе из аудиосигнала (v10.x Refactored)

Этот Jupyter Notebook представляет собой основной конвейер для обучения и оценки модели распознавания кода Морзе.

**Структура ноутбука:**

1.  **Импорты:** Загрузка всех необходимых библиотек.
2.  **Определение Корня Проекта:** Автоматическое определение базовой директории проекта.
3.  **Загрузка Конфигурации:** Чтение базового файла конфигурации (`experiment_base.json`).
4.  **Обновление Конфигурации и Утилиты:** Динамическое обновление конфига (устройство, калибровка), установка seed, настройка MLflow, очистка CUDA.
5.  **Подготовка Данных:** Загрузка CSV-файлов, создание словарей символов, формирование путей к аудио. Сканирование карт Перлина.
6.  **Разделение Данных:** Разделение основного датасета на обучающую и валидационную выборки.
7.  **Создание Аугментатора:** Инициализация объекта для аудио-аугментаций.
8.  **Основной Пайплайн Выполнения:**
- 8.1: Подготовка к запуску (определение режима, создание папки, старт MLflow).
- 8.2: Запуск обучения (`train_only`).
- 8.3: Запуск дообучения (`finetune_only`).
- 8.4: Генерация файла предсказаний (`submission.csv`).
- 8.5: Завершение пайплайна (сохранение финального конфига, завершение MLflow).
---

## 1. Импорты
---

In [1]:
# Ячейка 1: Импорты
# ------------------

# Стандартные библиотеки Python
import os
import gc
import sys
import json
import random
import time
import warnings
import math
import traceback
import contextlib
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

# Подавление ошибки Intel MKL (если необходимо)
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# Основные библиотеки Data Science
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import Levenshtein # pip install python-Levenshtein

# PyTorch и связанные библиотеки
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR # Оставляем только OneCycleLR
import torchaudio # Для SpecAugment и др.

# Аугментации Аудио
import audiomentations # pip install audiomentations

# Утилиты и Логирование
from tqdm.notebook import tqdm # Или from tqdm import tqdm для скриптов
import mlflow # pip install mlflow

# Игнорирование предупреждений (использовать с осторожностью)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- Вывод версий для воспроизводимости ---
print("--- Версии ключевых библиотек ---")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchaudio: {torchaudio.__version__}")
print(f"Librosa: {librosa.__version__}")
print(f"Audiomentations: {audiomentations.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"MLflow: {mlflow.__version__}")
print(f"Levenshtein: {'Доступен' if 'Levenshtein' in sys.modules else 'Не найден!'}")
print("-" * 30)

# --- Проверка доступности CUDA ---
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Устройств CUDA: {torch.cuda.device_count()}")
    print(f"Текущее устройство CUDA: {torch.cuda.current_device()} ({torch.cuda.get_device_name(torch.cuda.current_device())})")
print("-" * 30)
print("Ячейка 1 (Импорты) выполнена.")

--- Версии ключевых библиотек ---
Python: 3.9.21
PyTorch: 2.5.1+cu121
Torchaudio: 2.5.1+cu121
Librosa: 0.10.2.post1
Audiomentations: 0.40.0
Numpy: 2.0.2
Pandas: 2.2.3
MLflow: 2.21.3
Levenshtein: Доступен
------------------------------
CUDA доступна: True
Устройств CUDA: 1
Текущее устройство CUDA: 0 (NVIDIA GeForce GTX 1050 Ti)
------------------------------
Ячейка 1 (Импорты) выполнена.


## 2. Определение Корня Проекта

Находим корневую директорию проекта и добавляем ее в `sys.path` для импорта модулей из `src`.

---


In [2]:
# Ячейка 2: Определение Корня Проекта и Добавление в sys.path
# ---------------------------------------------------------

PROJECT_ROOT: Optional[Path] = None

try:
    # Пытаемся определить корень проекта
    current_working_dir = Path(os.getcwd()).resolve()
    potential_root = None

    # Сценарий 1: Мы в корне проекта (есть папка src)
    if (current_working_dir / 'src').is_dir():
        potential_root = current_working_dir
    # Сценарий 2: Мы в папке 'notebooks' (папка 'src' на уровень выше)
    elif (current_working_dir.parent / 'src').is_dir():
        potential_root = current_working_dir.parent
    # Сценарий 3: Мы где-то еще, ищем вверх по дереву
    else:
        temp_path = current_working_dir
        for _ in range(3): # Ищем не более 3 уровней вверх
            if (temp_path / 'src').is_dir():
                potential_root = temp_path
                break
            if temp_path == temp_path.parent: break # Дошли до корня диска
            temp_path = temp_path.parent

    if potential_root and (potential_root / 'src').is_dir():
         PROJECT_ROOT = potential_root
         print(f"Обнаружен корень проекта: {PROJECT_ROOT}")
    else:
        # Запасной вариант: предполагаем, что мы на один уровень ниже корня
        PROJECT_ROOT = current_working_dir.parent
        print(f"Предупреждение: Не удалось надежно определить корень проекта по папке 'src'.")
        print(f"Предполагаемый корень (родитель текущей папки): {PROJECT_ROOT}")
        if not (PROJECT_ROOT / 'README.md').exists() and not (PROJECT_ROOT / 'config').exists():
             print(f"!!! ВНИМАНИЕ: Предполагаемый корень {PROJECT_ROOT} может быть неверным!")

    # Добавляем корень проекта в sys.path, если его там еще нет
    project_root_str = str(PROJECT_ROOT)
    if project_root_str not in sys.path:
        sys.path.insert(0, project_root_str) # Добавляем в начало
        print(f"Добавлен '{project_root_str}' в sys.path.")
    else:
        print(f"'{project_root_str}' уже находится в sys.path.")

    # --- Импорты из src (после добавления в sys.path) ---
    from src.utils.common import set_seed
    from src.utils.mlflow_utils import setup_mlflow, log_mlflow_params # Используем обновленную утилиту
    from src.training.pipeline import run_training
    from src.data_processing.text import create_char_map
    from src.utils.path_utils import create_full_path
    from src.utils.file_system import find_available_perlin_maps
    from src.data_processing.augmentations import create_audio_augmenter
    from src.inference.predict import generate_submission

except ImportError as e_import_src:
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Не удалось импортировать модуль из 'src': {e_import_src}")
     print("   Убедитесь, что корень проекта определен верно и файлы в 'src' существуют.")
     raise
except Exception as e_proj_root:
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при определении корня проекта: {e_proj_root}")
     traceback.print_exc(limit=1)
     raise

print("\n--- Ячейка 2 (Определение Корня Проекта и Импорты src) завершена ---")

Обнаружен корень проекта: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder
Добавлен 'C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder' в sys.path.

--- Ячейка 2 (Определение Корня Проекта и Импорты src) завершена ---


## 3. Загрузка Базовой Конфигурации

Загружаем основной файл конфигурации `experiment_base.json` из папки `config`.

---

In [3]:
# Ячейка 3: Загрузка Базовой Конфигурации
# ---------------------------------------

CONFIG: Dict[str, Any] = {} # Словарь для хранения конфигурации
CONFIG_PATH = PROJECT_ROOT / 'config' / 'experiment_base.json'

print(f"Попытка загрузки базовой конфигурации из: {CONFIG_PATH}")

try:
    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(f"Файл конфигурации не найден: {CONFIG_PATH}")

    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        CONFIG = json.load(f)
    print(f"Базовая конфигурация ({CONFIG_PATH.name}) успешно загружена.")

    # Проверка наличия ключевых секций (пример)
    required_sections = ["paths", "audio", "model", "training", "finetuning", "ctc", "random_seed"]
    missing_sections = [s for s in required_sections if s not in CONFIG]
    if missing_sections:
        warnings.warn(f"Секции {missing_sections} отсутствуют в базовой конфигурации!")

except FileNotFoundError as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: {e}")
    print("   Убедитесь, что файл конфигурации существует и путь к нему верен.")
    raise
except json.JSONDecodeError as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Не удалось разобрать JSON файл конфигурации: {e}")
    print(f"   Проверьте синтаксис файла: {CONFIG_PATH}")
    raise
except Exception as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при загрузке конфигурации: {e}")
    traceback.print_exc(limit=1)
    raise

print("\n--- Ячейка 3 (Загрузка Конфигурации) завершена ---")

Попытка загрузки базовой конфигурации из: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\config\experiment_base.json
Базовая конфигурация (experiment_base.json) успешно загружена.

--- Ячейка 3 (Загрузка Конфигурации) завершена ---


## 4. Настройка Утилит

Устанавливаем random seed, определяем устройство, настраиваем MLflow и очищаем кэш CUDA.

---

In [4]:
# Ячейка 4: Настройка Утилит 
# ------------------------------------------

IS_MLFLOW_ACTIVE: bool = False # Флаг статуса MLflow

try:
    # --- 1. Определение Устройства ---
    print("--- Определение устройства ---")
    if CONFIG.get("device", "auto") == "auto":
        CONFIG["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    elif CONFIG["device"] == "cuda" and not torch.cuda.is_available():
        print("Предупреждение: Устройство в конфиге 'cuda', но CUDA недоступна. Используется 'cpu'.")
        CONFIG["device"] = "cpu"
    DEVICE = torch.device(CONFIG["device"]) # Сохраняем как объект torch.device
    print(f"Используемое устройство: {DEVICE}")

    # --- 1.1 Вычисление model.freq_dim --- <<< ДОБАВЛЕНО ЗДЕСЬ
    print("\n--- Вычисление model.freq_dim ---")
    try:
        n_fft = CONFIG.get("audio", {}).get("n_fft")
        if n_fft is not None and isinstance(n_fft, int) and n_fft > 0:
            # Убедимся, что секция model существует
            if "model" not in CONFIG: CONFIG["model"] = {}
            CONFIG["model"]["freq_dim"] = n_fft // 2 + 1 # <-- Вот оно!
            print(f"Вычислено model.freq_dim: {CONFIG['model']['freq_dim']} (из n_fft={n_fft})")
        else:
            # Если n_fft нет или некорректен, модель не сможет создаться. Прерываем выполнение.
            raise ValueError(f"Ключ 'audio.n_fft' (целое > 0) необходим для вычисления 'model.freq_dim', но отсутствует или некорректен в CONFIG (значение: {n_fft}).")
    except KeyError as e:
        print(f"❌ ОШИБКА: Отсутствует ключ '{e}' при доступе к CONFIG['model'] при вычислении freq_dim.")
        raise
    # --- КОНЕЦ ДОБАВЛЕННОГО БЛОКА ---

    # --- 2. Установка Random Seed ---
    seed = CONFIG.get("random_seed")
    if seed is None:
        warnings.warn("Ключ 'random_seed' не найден в CONFIG. Воспроизводимость не гарантирована.")
    else:
        print(f"\n--- Установка Random Seed: {seed} ---")
        set_seed(seed)
        print("Seed установлен для Python, Numpy и PyTorch.")

    # --- 3. Настройка MLflow ---
    print("\n--- Настройка MLflow ---")
    # Функция setup_mlflow должна брать параметры из CONFIG
    IS_MLFLOW_ACTIVE = setup_mlflow(CONFIG, project_root=PROJECT_ROOT)
    print(f"Статус MLflow после настройки: {'Активен' if IS_MLFLOW_ACTIVE else 'Неактивен'}")

    # --- 4. Очистка CUDA Cache ---
    if DEVICE.type == 'cuda':
        print("\n--- Очистка кэша CUDA ---")
        torch.cuda.empty_cache()
        gc.collect()
        print("Кэш CUDA очищен.")

except KeyError as ke:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Ключ '{ke}' не найден в CONFIG.")
    traceback.print_exc(limit=1)
    raise
except ValueError as ve: # Ловим ValueError от проверки n_fft
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: {ve}")
     raise
except NameError as ne:
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Переменная не определена - {ne}. Выполните предыдущие ячейки.")
     raise
except Exception as e_cell4:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА в Ячейке 4: {e_cell4}")
    traceback.print_exc(limit=2)
    raise

print("\n--- Ячейка 4 (Настройка Утилит) завершена ---")

--- Определение устройства ---
Используемое устройство: cuda

--- Вычисление model.freq_dim ---
Вычислено model.freq_dim: 257 (из n_fft=512)

--- Установка Random Seed: 42 ---
Seed установлен: 42
Seed установлен для Python, Numpy и PyTorch.

--- Настройка MLflow ---
MLflow Tracking URI не указан, используется локальное логирование (папка 'mlruns').
MLflow Experiment 'MorseCodeRecognition_v10_Final' установлен.
✅ MLflow успешно настроен и активен.
Статус MLflow после настройки: Активен

--- Очистка кэша CUDA ---
Кэш CUDA очищен.

--- Ячейка 4 (Настройка Утилит) завершена ---


## 5. Подготовка Данных

Загружаем метаданные, создаем словари символов, формируем пути к аудиофайлам и сканируем карты Перлина.

---

In [5]:
# Ячейка 5: Подготовка Данных
# --------------------------

# --- Глобальные переменные для данных ---
train_df_full: Optional[pd.DataFrame] = None
test_df: Optional[pd.DataFrame] = None
char_to_int: Dict[str, int] = {}
int_to_char: Dict[int, str] = {}
vocab_size: int = 0
AVAILABLE_PERLIN_INDICES: List[int] = []

try:
    # --- Получение параметров из CONFIG ---
    print("--- Загрузка параметров путей и столбцов из CONFIG ---")
    paths_cfg = CONFIG.get('paths', {})
    ctc_cfg = CONFIG.get('ctc', {})
    model_cfg = CONFIG.get('model', {}) # Получаем секцию model

    TRAIN_FILE_COLUMN = CONFIG['train_file_column']
    TEST_FILE_COLUMN = CONFIG['test_file_column']
    MORSE_CODE_COLUMN = CONFIG['morse_code_column']
    DATA_ROOT_STR = paths_cfg.get('data_dir', 'data')
    AUDIO_FOLDER_NAME = paths_cfg.get('audio_folder_name', 'morse_dataset/morse_dataset')
    PERLIN_MAPS_DIR_STR = paths_cfg.get("generic_perlin_maps_dir")

    # --- Формирование путей к данным ---
    data_root_path = PROJECT_ROOT / DATA_ROOT_STR
    raw_data_path = data_root_path / 'raw'
    train_csv_path = raw_data_path / "train.csv"
    test_csv_path = raw_data_path / "sample_submission.csv"
    audio_folder_path = raw_data_path / AUDIO_FOLDER_NAME

    print(f"Ожидаемый путь к train.csv: {train_csv_path.resolve()}")
    print(f"Ожидаемый путь к sample_submission.csv: {test_csv_path.resolve()}")
    print(f"Ожидаемый путь к аудиофайлам: {audio_folder_path.resolve()}")

    # --- Загрузка CSV ---
    print("\n--- Загрузка CSV файлов ---")
    if not train_csv_path.is_file(): raise FileNotFoundError(f"Файл train.csv не найден: {train_csv_path}")
    if not test_csv_path.is_file(): raise FileNotFoundError(f"Файл sample_submission.csv не найден: {test_csv_path}")

    train_df_full = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    print(f"Загружено: train.csv ({len(train_df_full)} строк), sample_submission.csv ({len(test_df)} строк)")

    # Проверка необходимых столбцов
    required_train_cols = [TRAIN_FILE_COLUMN, MORSE_CODE_COLUMN]
    required_test_cols = [TEST_FILE_COLUMN]
    if not all(col in train_df_full.columns for col in required_train_cols):
        raise ValueError(f"Столбцы {required_train_cols} должны быть в train.csv!")
    if not all(col in test_df.columns for col in required_test_cols):
        raise ValueError(f"Столбец {TEST_FILE_COLUMN} должен быть в sample_submission.csv!")

    # --- Создание словарей символов ---
    print("\n--- Создание словаря символов ---")
    char_to_int, int_to_char, vocab_size = create_char_map(
        train_df_full[MORSE_CODE_COLUMN].astype(str).tolist(), ctc_cfg
    )
    # Сохраняем размер словаря в конфиг модели
    model_cfg["vocab_size"] = vocab_size
    print(f"Словарь создан. Размер (включая спец. символы): {vocab_size}")

    # --- Формирование полных путей к аудиофайлам ---
    print("\n--- Формирование полных путей к аудиофайлам ---")
    if not audio_folder_path.is_dir():
         warnings.warn(f"Папка с аудио {audio_folder_path} не найдена! Проверьте путь и конфиг.")

    train_df_full['full_path'] = train_df_full[TRAIN_FILE_COLUMN].apply(lambda x: create_full_path(x, audio_folder_path))
    test_df['full_path'] = test_df[TEST_FILE_COLUMN].apply(lambda x: create_full_path(x, audio_folder_path))
    print("Полные пути добавлены в DataFrame'ы.")

    # --- Сканирование карт Перлина ---
    print("\n--- Сканирование доступных карт Перлина ---")
    if PERLIN_MAPS_DIR_STR:
        perlin_maps_dir = PROJECT_ROOT / PERLIN_MAPS_DIR_STR
        print(f"Поиск карт в: {perlin_maps_dir.resolve()}")
        AVAILABLE_PERLIN_INDICES = find_available_perlin_maps(perlin_maps_dir)
        num_maps = len(AVAILABLE_PERLIN_INDICES)
        if not AVAILABLE_PERLIN_INDICES:
            print("Предупреждение: Не найдено доступных карт Перлина.")
        else:
            print(f"Найдено {num_maps} карт Перлина.")
        # Сохраняем количество карт в информацию о запуске (даже если 0)
        CONFIG.setdefault("info", {})["num_available_perlin_maps"] = num_maps
    else:
        print("Путь к картам Перлина ('generic_perlin_maps_dir') не указан в конфиге.")
        CONFIG.setdefault("info", {})["num_available_perlin_maps"] = 0

except FileNotFoundError as e: print(f"❌ ОШИБКА: Файл не найден - {e}"); raise
except ValueError as e: print(f"❌ ОШИБКА: Некорректное значение или столбец - {e}"); raise
except KeyError as e: print(f"❌ ОШИБКА: Ключ не найден в CONFIG - {e}"); raise
except Exception as e_cell5: print(f"❌ КРИТИЧЕСКАЯ ОШИБКА в Ячейке 5: {e_cell5}"); traceback.print_exc(limit=2); raise

print("\n--- Ячейка 5 (Подготовка Данных) завершена ---")

--- Загрузка параметров путей и столбцов из CONFIG ---
Ожидаемый путь к train.csv: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw\train.csv
Ожидаемый путь к sample_submission.csv: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw\sample_submission.csv
Ожидаемый путь к аудиофайлам: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw\data\raw\morse_dataset\morse_dataset

--- Загрузка CSV файлов ---
Загружено: train.csv (30000 строк), sample_submission.csv (5000 строк)

--- Создание словаря символов ---
Найдено уникальных символов в текстах (44):  #0123456789АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ
Размер словаря (Vocab Size, включая бланк): 45
Словарь (idx: char): {0: '<blank>', 1: ' ', 2: '#', 3: '0', 4: '1', 5: '2', 6: '3', 7: '4', 8: '5', 9: '6', 10: '7', 11: '8', 12: '9', 13: 'А', 14: 'Б', 15: 'В', 16: 'Г', 17: 'Д', 18: 'Е', 19: 'Ж', 20: 'З', 21: 'И', 22: 'Й', 23: 'К', 24: 'Л', 25: 'М', 26: 'Н', 27: 'О', 28: 'П', 29: 'Р', 30: 'С', 31: 'Т', 32: 'У', 

## 6. Разделение Данных на Обучение и Валидацию

Разделяем `train_df_full` на обучающий и валидационный наборы.

---

In [6]:
# Ячейка 6: Разделение Данных на Обучение и Валидацию
# -------------------------------------------------

train_split_df: Optional[pd.DataFrame] = None
val_split_df: Optional[pd.DataFrame] = None

try:
    print("--- Разделение данных на Train/Validation ---")
    if train_df_full is None or train_df_full.empty:
        raise ValueError("DataFrame 'train_df_full' не определен или пуст. Выполните Ячейку 5.")

    # Копируем для работы
    working_df = train_df_full.copy().reset_index(drop=True)

    # --- Выполнение разделения ---
    val_split_ratio = CONFIG.get("training", {}).get("val_split_ratio", 0.1)
    if not (0 < val_split_ratio < 1):
        raise ValueError(f"Некорректный val_split_ratio ({val_split_ratio}). Должен быть между 0 и 1.")

    val_size = int(len(working_df) * val_split_ratio)
    train_size = len(working_df) - val_size

    if val_size <= 0 or train_size <= 0:
        raise ValueError(f"Некорректные размеры после разделения: Train={train_size}, Val={val_size}.")

    print(f"Разделение {len(working_df)} записей на Train ({train_size}) и Val ({val_size}) с Seed: {CONFIG['random_seed']}")

    # Используем генератор PyTorch для воспроизводимого разделения
    generator = torch.Generator().manual_seed(CONFIG["random_seed"])
    train_indices, val_indices = random_split(range(len(working_df)), [train_size, val_size], generator=generator)

    # Создаем финальные DataFrame'ы
    train_split_df = working_df.iloc[train_indices.indices].copy().reset_index(drop=True)
    val_split_df = working_df.iloc[val_indices.indices].copy().reset_index(drop=True)

    print(f"Данные успешно разделены: Train={len(train_split_df)}, Val={len(val_split_df)}")

except NameError as ne: print(f"❌ ОШИБКА: Переменная 'train_df_full' не определена. {ne}"); raise
except ValueError as ve: print(f"❌ ОШИБКА при разделении данных: {ve}"); raise
except KeyError as ke: print(f"❌ ОШИБКА: Ключ не найден в CONFIG - {ke}"); raise
except Exception as e_split: print(f"❌ КРИТИЧЕСКАЯ ОШИБКА на этапе разделения данных: {e_split}"); traceback.print_exc(limit=2); raise

print("\n--- Ячейка 6 (Разделение Данных) завершена ---")

--- Разделение данных на Train/Validation ---
Разделение 30000 записей на Train (27000) и Val (3000) с Seed: 42
Данные успешно разделены: Train=27000, Val=3000

--- Ячейка 6 (Разделение Данных) завершена ---


## 7. Создание Аудио-Аугментатора

Инициализируем объект для аудио-аугментаций, если они включены в конфигурации.

---

In [7]:
# Ячейка 7: Создание Аудио-Аугментатора
# ------------------------------------

AUDIO_AUGMENTER_GLOBAL: Optional[audiomentations.Compose] = None

try:
    print("--- Создание объекта аудио-аугментатора ---")
    audio_aug_config = CONFIG.get("audio_augmentation", {})
    if audio_aug_config.get("apply", False):
        # Функция create_audio_augmenter должна быть в src/data_processing/augmentations.py
        AUDIO_AUGMENTER_GLOBAL = create_audio_augmenter(CONFIG)
        if AUDIO_AUGMENTER_GLOBAL:
            print("Глобальный объект аудио-аугментатора успешно создан.")
        else:
            print("Предупреждение: Аудио-аугментатор не создан (возможно, pipeline пуст).")
            audio_aug_config["apply"] = False # Убедимся, что флаг отключен
    else:
        print("Аудио-аугментации НЕ будут применяться (apply=False в конфиге).")

except KeyError as ke: print(f"❌ ОШИБКА: Ключ не найден в CONFIG при создании аугментатора - {ke}"); raise
except Exception as e_aug:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при создании аудио-аугментатора: {e_aug}")
    traceback.print_exc(limit=2)
    AUDIO_AUGMENTER_GLOBAL = None
    if "audio_augmentation" in CONFIG: CONFIG["audio_augmentation"]["apply"] = False

print("\n--- Ячейка 7 (Создание Аугментатора) завершена ---")

--- Создание объекта аудио-аугментатора ---

--- Создание конвейера Аудио-Аугментаций ---
  + Аудио-Ауг: AddGaussianNoise (p=0.8)
  + Аудио-Ауг: Gain (p=0.8)
  + Аудио-Ауг: ClippingDistortion (p=0.7)
Глобальный объект аудио-аугментатора успешно создан.

--- Ячейка 7 (Создание Аугментатора) завершена ---


## 8. Основной Пайплайн Выполнения

Оркестрируем процесс обучения (train_only) или дообучения (finetune_only) и генерации предсказаний.

---


### 8.1 Подготовка к Запуску и Старт MLflow

Определяем режим, создаем директорию для результатов, сохраняем начальный конфиг, стартуем MLflow run.



In [8]:
# Ячейка 8.1: Подготовка к Запуску и Старт MLflow
# -----------------------------------------------

# --- Проверка наличия необходимых переменных ---
required_vars_stage8 = [
    'CONFIG', 'PROJECT_ROOT', 'train_split_df', 'val_split_df', 'test_df',
    'char_to_int', 'int_to_char', 'AUDIO_AUGMENTER_GLOBAL',
    'AVAILABLE_PERLIN_INDICES', 'IS_MLFLOW_ACTIVE', 'DEVICE',
    'run_training', 'generate_submission', 'log_mlflow_params' # Функции
]
for var in required_vars_stage8:
    if var not in locals() and var not in globals():
        raise NameError(f"Переменная или функция '{var}' не определена. Выполните предыдущие ячейки.")

# --- Определение режима работы и директории запуска ---
run_mode = CONFIG.get("run_mode", "train_only") # По умолчанию train_only
if run_mode not in ["train_only", "finetune_only"]:
    raise ValueError(f"Неподдерживаемый run_mode в конфиге: '{run_mode}'. Допустимы 'train_only' или 'finetune_only'.")

run_description_safe = CONFIG.get('run_description', 'default_run').replace(":", "-").replace(" ", "_").replace("/", "_")
# Добавляем режим к имени папки для ясности
run_dir_name = f"{run_description_safe}_{run_mode}"
OUTPUT_DIR_RUN = PROJECT_ROOT / CONFIG["paths"]["output_dir"] / run_dir_name

# --- Инициализация переменных для результатов ---
final_model_path: str = ""
final_levenshtein: float = float('inf')
pipeline_start_time = time.time()
active_mlflow_run_id: Optional[str] = None
pipeline_status: str = "STARTED" # Статус пайплайна

print("\n" + "="*50)
print(f" ЗАПУСК ОСНОВНОГО КОНВЕЙЕРА (Режим: {run_mode}) ")
print(f" Директория для результатов этого запуска: {OUTPUT_DIR_RUN.resolve()}")
print("="*50)

try:
    # --- Создание выходной директории ЗАПУСКА ---
    OUTPUT_DIR_RUN.mkdir(parents=True, exist_ok=True)
    print(f"Выходная директория ЗАПУСКА создана/проверена.")

    # --- Сохранение НАЧАЛЬНОГО конфига ЗАПУСКА ---
    initial_config_save_path = OUTPUT_DIR_RUN / f"config_initial_{run_dir_name}.json"
    initial_config_snapshot = CONFIG.copy()
    # Добавляем информацию о запуске
    initial_config_snapshot['info'] = initial_config_snapshot.get('info', {})
    initial_config_snapshot['info'].update({
        'python_version': sys.version.split()[0],
        'torch_version': torch.__version__,
        'start_time_utc': time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        'project_root': str(PROJECT_ROOT),
        'output_dir_run': str(OUTPUT_DIR_RUN),
        'actual_run_mode': run_mode,
        'device_used': str(DEVICE)
    })
    with open(initial_config_save_path, 'w', encoding='utf-8') as f:
        json.dump(initial_config_snapshot, f, indent=4, ensure_ascii=False, default=str)
    print(f"Начальная конфигурация ЗАПУСКА сохранена: {initial_config_save_path.name}")

    # --- Старт MLflow Run ---
    if IS_MLFLOW_ACTIVE:
        print("\n--- Старт MLflow Run ---")
        try:
            # Используем имя папки как имя запуска для уникальности
            active_run = mlflow.start_run(run_name=run_dir_name)
            active_mlflow_run_id = active_run.info.run_id
            print(f"MLflow Run начат. ID: {active_mlflow_run_id}")
            # Логируем начальный конфиг
            mlflow.log_artifact(str(initial_config_save_path), artifact_path="config")
            # Логируем ключевые параметры запуска
            log_mlflow_params(initial_config_snapshot.get('info', {}), prefix="info")
            log_mlflow_params(CONFIG.get('audio', {}), prefix="audio")
            log_mlflow_params(CONFIG.get('model', {}), prefix="model")
            log_mlflow_params(CONFIG.get(run_mode[:-5]+'ing', {}), prefix=run_mode[:-5]) # train или finetune
            print("Начальный конфиг и параметры залогированы в MLflow.")
        except Exception as e_mlflow_start:
            print(f"⚠️ Ошибка при старте или логировании в MLflow: {e_mlflow_start}")
            print("   MLflow будет считаться неактивным для этого запуска.")
            IS_MLFLOW_ACTIVE = False
            active_mlflow_run_id = None
    else:
        print("\nMLflow неактивен, логирование пропускается.")

except Exception as e_prep:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА на этапе подготовки к запуску: {e_prep}")
    traceback.print_exc(limit=2)
    pipeline_status = "ERROR_PREP"
    # Попытка завершить MLflow run, если он был начат
    if active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
        mlflow.set_tag("pipeline_error_stage", "preparation")
        mlflow.end_run(status="FAILED")
        print("MLflow run завершен со статусом FAILED.")
    raise # Прерываем выполнение ноутбука

print("\n--- Ячейка 8.1 (Подготовка и Старт MLflow) завершена ---")


 ЗАПУСК ОСНОВНОГО КОНВЕЙЕРА (Режим: finetune_only) 
 Директория для результатов этого запуска: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\outputs\CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Final_finetune_only
Выходная директория ЗАПУСКА создана/проверена.
Начальная конфигурация ЗАПУСКА сохранена: config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Final_finetune_only.json

--- Старт MLflow Run ---
MLflow Run начат. ID: ee1f8f73db294937b0ceccf19883c3a9
Начальный конфиг и параметры залогированы в MLflow.

--- Ячейка 8.1 (Подготовка и Старт MLflow) завершена ---


### 8.2 Запуск Обучения (`train_only`)

Выполняется только если `run_mode` == `"train_only"`.

In [9]:
# Ячейка 8.2: Запуск Обучения ('train_only')
# ------------------------------------------

best_train_model_path = ""
best_train_lev = float('inf')

if run_mode == "train_only" and pipeline_status == "STARTED":
    print("\n" + "="*20 + " Этап: Основное Обучение ('train_only') " + "="*20)
    stage_start_time = time.time()
    try:
        # Вызываем run_training для режима 'train'
        best_train_model_path, best_train_lev = run_training(
            mode='train',
            config=CONFIG,
            train_df=train_split_df,
            val_df=val_split_df,
            char_to_int=char_to_int,
            int_to_char=int_to_char,
            audio_augmenter_global=AUDIO_AUGMENTER_GLOBAL,
            available_map_indices=AVAILABLE_PERLIN_INDICES,
            IS_MLFLOW_ACTIVE=IS_MLFLOW_ACTIVE,
            project_root=PROJECT_ROOT,
            output_dir_run=OUTPUT_DIR_RUN # Передаем папку запуска
        )

        if best_train_model_path and Path(best_train_model_path).exists() and np.isfinite(best_train_lev):
            final_model_path = best_train_model_path
            final_levenshtein = best_train_lev
            print(f"\n✅ Основное обучение ('train') завершено УСПЕШНО.")
            print(f"   Лучший Levenshtein (Val): {final_levenshtein:.4f}")
            print(f"   Модель сохранена: {Path(final_model_path).name}")
            pipeline_status = "TRAIN_COMPLETED"
        else:
            print("\n⚠️ Основное обучение ('train') завершилось БЕЗ сохранения лучшей модели.")
            pipeline_status = "TRAIN_FAILED_NO_MODEL"
            # Не устанавливаем final_model_path, чтобы submission не запускался

    except Exception as e_train:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА во время основного обучения ('train'): {e_train}")
        traceback.print_exc(limit=2)
        pipeline_status = "ERROR_DURING_TRAIN"
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.set_tag("pipeline_error_stage", "train")
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None # Сбрасываем ID, чтобы не пытаться завершить снова

    stage_duration = time.time() - stage_start_time
    print(f"--- Этап 'train_only' занял: {stage_duration:.2f} сек. ---")

elif run_mode != "train_only":
    print(f"\n--- Этап: Основное Обучение ('train_only') ПРОПУЩЕН (Режим: {run_mode}) ---")
elif pipeline_status != "STARTED":
     print(f"\n--- Этап: Основное Обучение ('train_only') ПРОПУЩЕН из-за предыдущей ошибки ({pipeline_status}) ---")

print(f"\n--- Ячейка 8.2 (Основное Обучение) завершена (Статус: {pipeline_status}) ---")


--- Этап: Основное Обучение ('train_only') ПРОПУЩЕН (Режим: finetune_only) ---

--- Ячейка 8.2 (Основное Обучение) завершена (Статус: STARTED) ---


### 8.3 Запуск Дообучения (`finetune_only`)
Выполняется только если `run_mode` == `"finetune_only"`.

In [ ]:
# Ячейка 8.3: Запуск Дообучения ('finetune_only')
# ---------------------------------------------

best_ft_model_path = ""
best_ft_lev = float('inf')

if run_mode == "finetune_only" and pipeline_status == "STARTED":
    print("\n" + "="*20 + " Этап: Дообучение ('finetune_only') " + "="*20)
    stage_start_time = time.time()
    try:
        # --- Получение и проверка пути к базовой модели ---
        finetuning_cfg = CONFIG.get("finetuning", {})
        checkpoint_path_rel = finetuning_cfg.get("finetune_only_checkpoint_path")
        if not checkpoint_path_rel:
            raise ValueError("В режиме 'finetune_only' не указан путь 'finetuning.finetune_only_checkpoint_path' в CONFIG!")

        checkpoint_file = Path(checkpoint_path_rel)
        if not checkpoint_file.is_absolute():
            # Ищем относительно корня проекта или папки outputs (более гибко)
            potential_paths = [
                PROJECT_ROOT / checkpoint_path_rel,
                PROJECT_ROOT / CONFIG["paths"]["output_dir"] / checkpoint_path_rel
            ]
            found = False
            for p in potential_paths:
                if p.exists():
                    checkpoint_file = p.resolve()
                    found = True
                    break
            if not found:
                 raise FileNotFoundError(f"Не найден файл модели для finetune_only: '{checkpoint_path_rel}' (проверено в {PROJECT_ROOT} и {PROJECT_ROOT / CONFIG['paths']['output_dir']})")
        elif not checkpoint_file.exists():
             raise FileNotFoundError(f"Не найден файл модели для finetune_only: {checkpoint_file.resolve()}")

        absolute_checkpoint_path = str(checkpoint_file)
        print(f"Используется базовая модель для дообучения: {checkpoint_file.name}")
        # Попытка извлечь Levenshtein из имени файла (не критично)
        try:
            match = re.search(r"lev([\d.]+)", checkpoint_file.name)
            base_lev = float(match.group(1)) if match else float('inf')
            print(f"  (Предполагаемый Lev из имени файла: {base_lev:.4f if np.isfinite(base_lev) else 'N/A'})")
        except Exception: pass

        # --- Вызов run_training для режима 'finetune' ---
        best_ft_model_path, best_ft_lev = run_training(
            mode='finetune',
            config=CONFIG,
            train_df=train_split_df,
            val_df=val_split_df,
            char_to_int=char_to_int,
            int_to_char=int_to_char,
            audio_augmenter_global=AUDIO_AUGMENTER_GLOBAL,
            available_map_indices=AVAILABLE_PERLIN_INDICES,
            IS_MLFLOW_ACTIVE=IS_MLFLOW_ACTIVE,
            project_root=PROJECT_ROOT,
            output_dir_run=OUTPUT_DIR_RUN # Передаем папку запуска
            # checkpoint_path передается внутри run_training на основе конфига
        )

        if best_ft_model_path and Path(best_ft_model_path).exists() and np.isfinite(best_ft_lev):
            final_model_path = best_ft_model_path
            final_levenshtein = best_ft_lev
            print(f"\n✅ Дообучение ('finetune') завершено УСПЕШНО.")
            print(f"   Лучший Levenshtein FT: {final_levenshtein:.4f}")
            print(f"   Модель FT сохранена: {Path(final_model_path).name}")
            pipeline_status = "FINETUNE_COMPLETED"
        else:
            print("\n⚠️ Дообучение ('finetune') завершилось БЕЗ сохранения лучшей модели.")
            pipeline_status = "FINETUNE_FAILED_NO_MODEL"
            # Если дообучение не дало лучшей модели, стоит ли использовать исходную?
            # Пока оставляем final_model_path пустым, чтобы submission не генерировался.
            # Можно изменить логику, если нужно использовать исходную модель.

    except (ValueError, FileNotFoundError) as e_ft_prep:
         print(f"❌ ОШИБКА подготовки к дообучению: {e_ft_prep}")
         pipeline_status = "ERROR_PREP_FINETUNE"
    except Exception as e_finetune:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА во время дообучения ('finetune'): {e_finetune}")
        traceback.print_exc(limit=2)
        pipeline_status = "ERROR_DURING_FINETUNE"
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.set_tag("pipeline_error_stage", "finetune")
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None

    stage_duration = time.time() - stage_start_time
    print(f"--- Этап 'finetune_only' занял: {stage_duration:.2f} сек. ---")

elif run_mode != "finetune_only":
    print(f"\n--- Этап: Дообучение ('finetune_only') ПРОПУЩЕН (Режим: {run_mode}) ---")
elif pipeline_status != "STARTED":
     print(f"\n--- Этап: Дообучение ('finetune_only') ПРОПУЩЕН из-за предыдущей ошибки ({pipeline_status}) ---")

print(f"\n--- Ячейка 8.3 (Дообучение) завершена (Статус: {pipeline_status}) ---")


==================== Этап: Дообучение ('finetune_only') ====================
Используется базовая модель для дообучения: MorseCRNN_Final_TRAIN_epoch8_lev0.4070.pth

>> Запуск run_training (mode='finetune') <<
Режим дообучения. Базовая модель: MorseCRNN_Final_TRAIN_epoch8_lev0.4070.pth

--- Чтение параметров для этапа 'FINETUNE' ---
  Device: cuda, Epochs: 20, BS: 16, LR: 3.0e-04
  Optimizer: AdamW, Scheduler: OneCycleLR, AMP: True
  Augment (Stage): Audio=True
  Augment (Online): Perlin=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 1

Эпоха 1/20 [FINETUNE]:   0%|                                                                                  …

   [Валидация]:   0%|                                                                                         …


Epoch 1/20 Results | Duration: 1158.88 sec
  Train Loss: 1.3916
  Val Loss:   0.2604 | Val Levenshtein:   0.5590 (Best: inf)
  Current LR: 1.04e-04
  Примеры декодирования (Предсказание | Реальность):
    'ФААР834ОМП' | 'ФААР834ОМП'
    'Т0С ЖЩ0О БМЫ' | 'Т0С ЖЩ0О БМЫ'
    'ЬЛННЖЗУЯН3ЭХ7' | 'ЬЛННЖЗУЯН3ЭХ7'
  ✨ Val Levenshtein УЛУЧШИЛСЯ: inf -> 0.5590
  Модель СОХРАНЕНА как: MorseCRNN_Final_FINETUNE_epoch1_lev0.5590.pth
------------------------------------------------------------
Epoch 2/20 (FINETUNE)


Эпоха 2/20 [FINETUNE]:   0%|                                                                                  …

...


Epoch 7/20 Results | Duration: 911.96 sec
  Train Loss: 0.2921
  Val Loss:   0.1465 | Val Levenshtein:   0.3277 (Best: 0.3390)
  Current LR: 2.56e-04
  Примеры декодирования (Предсказание | Реальность):
    'ФААР834ОМП' | 'ФААР834ОМП'
    'Т0С ЖЩ0О БМЫ' | 'Т0С ЖЩ0О БМЫ'
    'ЬЛННЖЗУЯН3ЭХ7' | 'ЬЛННЖЗУЯН3ЭХ7'
  ✨ Val Levenshtein УЛУЧШИЛСЯ: 0.3390 -> 0.3277
  Модель СОХРАНЕНА как: MorseCRNN_Final_FINETUNE_epoch7_lev0.3277.pth
------------------------------------------------------------
Epoch 8/20 (FINETUNE)


Эпоха 8/20 [FINETUNE]:   0%|                                                                                  …

   [Валидация]:   0%|                                                                                         …


Epoch 8/20 Results | Duration: 882.56 sec
  Train Loss: 0.2752
  Val Loss:   0.1454 | Val Levenshtein:   0.3273 (Best: 0.3277)
  Current LR: 2.36e-04
  Примеры декодирования (Предсказание | Реальность):
    'ААР834ОМП' | 'ФААР834ОМП'
    'Т0С ЖЩ0О БМЫ' | 'Т0С ЖЩ0О БМЫ'
    'ЬЛННЖЗУЯН3ЭХ7' | 'ЬЛННЖЗУЯН3ЭХ7'
  ✨ Val Levenshtein УЛУЧШИЛСЯ: 0.3277 -> 0.3273
  Модель СОХРАНЕНА как: MorseCRNN_Final_FINETUNE_epoch8_lev0.3273.pth
------------------------------------------------------------
Epoch 9/20 (FINETUNE)


Эпоха 9/20 [FINETUNE]:   0%|                                                                                  …

### 8.4 Генерация Финального Submission
Генерируем `submission.csv`, если предыдущий этап завершился успешно и есть финальная модель.

In [ ]:
# Ячейка 8.4: Генерация Финального Submission
# -------------------------------------------

submission_generated_flag: bool = False
submission_file_name: str = "submission_default.csv"
submission_save_path: Optional[Path] = None

# Проверяем, что не было ошибок и есть путь к модели
should_generate_submission = ("ERROR" not in pipeline_status and
                              final_model_path and
                              Path(final_model_path).exists())

if should_generate_submission:
    print("\n" + "="*20 + " Этап: Генерация Финального Submission " + "="*20)
    stage_start_time = time.time()
    print(f"Используется финальная лучшая модель: {Path(final_model_path).name}")
    print(f"  Ее лучший Val Levenshtein: {f'{final_levenshtein:.4f}' if np.isfinite(final_levenshtein) else 'N/A'}")

    # Формируем имя файла submission
    lev_suffix = f"lev{f'{final_levenshtein:.4f}' if np.isfinite(final_levenshtein) else 'NA'}"
    
    # Используем имя папки запуска для уникальности
    submission_file_name = f"submission_{run_dir_name}_{lev_suffix}.csv"
    submission_save_path = OUTPUT_DIR_RUN / submission_file_name

    try:
        # Вызываем функцию генерации submission
        submission_generated_flag = generate_submission(
            config=CONFIG,
            model_path=final_model_path,
            test_df=test_df,
            char_to_int=char_to_int,
            int_to_char=int_to_char,
            output_dir_run=OUTPUT_DIR_RUN,
            submission_filename=submission_file_name,
            IS_MLFLOW_ACTIVE=IS_MLFLOW_ACTIVE,
            project_root=PROJECT_ROOT,
            device=DEVICE # Передаем устройство
        )
        if submission_generated_flag and submission_save_path.exists():
            print(f"✅ Файл Submission успешно сгенерирован: {submission_save_path.resolve()}")
            pipeline_status = "SUBMISSION_GENERATED"
            # Логирование артефакта в MLflow
            if IS_MLFLOW_ACTIVE and active_mlflow_run_id:
                 with contextlib.suppress(Exception):
                    mlflow.log_artifact(str(submission_save_path), artifact_path="submissions")
                    print("Файл Submission залогирован как артефакт в MLflow.")
        else:
            print("⚠️ Функция generate_submission сообщила об ошибке или файл не создан.")
            pipeline_status = "ERROR_SUBMISSION_GENERATION"

    except NameError as ne:
        print(f"❌ ОШИБКА: Переменная 'test_df' не определена. Выполните Ячейку 5.")
        pipeline_status = "ERROR_SUBMISSION_PREP"
        traceback.print_exc(limit=1)
    except Exception as e_sub:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА во время генерации submission: {e_sub}")
        traceback.print_exc(limit=2)
        submission_generated_flag = False
        pipeline_status = "ERROR_DURING_SUBMISSION"
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.set_tag("pipeline_error_stage", "submission")
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None

    stage_duration = time.time() - stage_start_time
    print(f"--- Этап 'submission' занял: {stage_duration:.2f} сек. ---")

elif "ERROR" not in pipeline_status:
    print("\n--- Этап: Генерация Submission ПРОПУЩЕНА ---")
    if not final_model_path: print("   Причина: Нет финальной модели после предыдущего этапа.")
    elif not Path(final_model_path).exists(): print(f"   Причина: Файл финальной модели не найден: {final_model_path}")
    else: print(f"   Причина: Неизвестная.")
else:
     print(f"\n--- Этап: Генерация Submission ПРОПУЩЕНА из-за предыдущей ошибки ({pipeline_status}) ---")

print(f"\n--- Ячейка 8.4 (Генерация Submission) завершена (Статус: {pipeline_status}) ---")


--- Этап: Генерация Submission ПРОПУЩЕНА ---
   Причина: Нет финальной модели после предыдущего этапа.

--- Ячейка 8.4 (Генерация Submission) завершена (Статус: FINETUNE_FAILED_NO_MODEL) ---


### 8.5 Завершение Пайплайна и MLflow
Сохраняем финальную конфигурацию с результатами и завершаем MLflow run.

In [ ]:
# Ячейка 8.5: Завершение Пайплайна и MLflow
# ----------------------------------------

print("\n" + "="*20 + " Этап: Завершение Пайплайна " + "="*20)

try:
    # --- Подготовка и сохранение финального конфига ---
    final_config_to_save = CONFIG.copy()
    pipeline_end_time = time.time()
    total_pipeline_duration = pipeline_end_time - pipeline_start_time

    # Собираем результаты пайплайна
    pipeline_results = {
        "final_status": pipeline_status,
        "run_mode_executed": run_mode,
        "final_best_model_path_relative": str(Path(final_model_path).relative_to(PROJECT_ROOT)) if final_model_path and Path(final_model_path).is_file() else "N/A",
        "final_best_model_name": Path(final_model_path).name if final_model_path else "N/A",
        "final_best_val_levenshtein": final_levenshtein if np.isfinite(final_levenshtein) else -1.0,
        "submission_generated": submission_generated_flag,
        "submission_filename": submission_file_name if submission_generated_flag else "N/A",
        "submission_path_relative": str(submission_save_path.relative_to(PROJECT_ROOT)) if submission_save_path and submission_save_path.is_file() else "N/A",
        "total_pipeline_duration_sec": round(total_pipeline_duration, 2),
        "mlflow_run_id": active_mlflow_run_id if active_mlflow_run_id else "N/A",
        "end_time_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    }
    # Добавляем результаты в секцию 'info' или создаем ее
    final_config_to_save.setdefault("info", {})["pipeline_results"] = pipeline_results

    # Сохраняем финальный конфиг
    final_config_filename = f"config_final_{run_dir_name}.json"
    final_config_save_path = OUTPUT_DIR_RUN / final_config_filename
    with open(final_config_save_path, 'w', encoding='utf-8') as f:
        json.dump(final_config_to_save, f, indent=4, ensure_ascii=False, default=str)
    print(f"Финальная конфигурация с результатами сохранена: {final_config_save_path.resolve()}")

    # --- Логирование финальных данных в MLflow (если активен и не завершен) ---
    if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
         try:
             # Логируем финальный конфиг
             mlflow.log_artifact(str(final_config_save_path), artifact_path="config")
             print("Финальный конфиг залогирован как артефакт в MLflow.")
             # Логируем финальные результаты как метрики/параметры
             log_mlflow_params(pipeline_results, prefix="results")
             if np.isfinite(final_levenshtein):
                 mlflow.log_metric("results_final_best_val_levenshtein", final_levenshtein) # Дублируем для графиков
             print(f"Финальные результаты залогированы в MLflow.")
         except Exception as e_art_cfg:
             print(f"Предупреждение: Не удалось залогировать финальный конфиг/результаты в MLflow: {e_art_cfg}")

except Exception as e_final_cfg:
    print(f"❌ Ошибка сохранения/логирования финального конфига: {e_final_cfg}")
    traceback.print_exc(limit=2)
    pipeline_status = "ERROR_FINALIZATION" # Обновляем статус

# --- Завершение MLflow run ---
final_mlflow_status = "FINISHED" if "ERROR" not in pipeline_status else "FAILED"
if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
    print(f"\nЗавершение финального MLflow run (ID: {active_mlflow_run_id}) со статусом {final_mlflow_status}...")
    mlflow.set_tag("final_pipeline_status", pipeline_status) # Добавляем тег с финальным статусом
    mlflow.end_run(status=final_mlflow_status)
    print("MLflow run завершен.")
elif active_mlflow_run_id:
     print("\nMLflow run был завершен ранее (вероятно, из-за ошибки).")
else:
     print("\nMLflow не был активен в этом запуске.")

# --- Финальный вывод ---
print("\n" + "="*50 + f"\n КОНВЕЙЕР ЗАВЕРШЕН (Итоговый статус: {pipeline_status}) \n" + "="*50)
print(f"Итоговое время выполнения: {total_pipeline_duration:.2f} сек. ({total_pipeline_duration/60:.2f} мин.)")

if "ERROR" not in pipeline_status and final_model_path and np.isfinite(final_levenshtein):
    print(f"\nФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
    print(f"  Режим выполнения: {run_mode}")
    print(f"  Лучшая модель: {Path(final_model_path).name}")
    print(f"  Лучший Levenshtein (Val): {final_levenshtein:.4f}")
    if submission_generated_flag: print(f"  Файл Submission: {submission_file_name} (в папке {OUTPUT_DIR_RUN.name})")
    else: print(f"  Файл Submission: НЕ СГЕНЕРИРОВАН или ошибка генерации")
elif "ERROR" in pipeline_status:
     print("\nПайплайн завершился с ОШИБКОЙ.")
     print(f"  Статус ошибки: {pipeline_status}")
     print(f"  Проверьте логи выше для деталей.")
else: # Случаи типа TRAIN_FAILED_NO_MODEL
    print("\nПайплайн завершился, но финальная модель не была успешно создана/сохранена.")
    print(f"  Статус: {pipeline_status}")

print("\n--- Ячейка 8.5 (Завершение Пайплайна) завершена ---")


==================== Этап: Завершение Пайплайна ====================
Финальная конфигурация с результатами сохранена: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\outputs\CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Final_finetune_only\config_final_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Final_finetune_only.json
Финальный конфиг залогирован как артефакт в MLflow.
Финальные результаты залогированы в MLflow.

Завершение финального MLflow run (ID: b82348a8f54d49948ac4885b306886b6) со статусом FINISHED...
MLflow run завершен.

 КОНВЕЙЕР ЗАВЕРШЕН (Итоговый статус: FINETUNE_FAILED_NO_MODEL) 
Итоговое время выполнения: 0.40 сек. (0.01 мин.)

Пайплайн завершился, но финальная модель не была успешно создана/сохранена.
  Статус: FINETUNE_FAILED_NO_MODEL

--- Ячейка 8.5 (Завершение Пайплайна) завершена ---
